In [1]:

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

import json
import jiwer #https://github.com/jitsi/jiwer
import time

import numpy as np

<h4> Data Loader </h4>

In [2]:
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\audios.json", "r", encoding = "utf-8") as f:
    json_file = json.load (f)

display (json_file)

"""
    Normalização do Dataset.
    É importante que tanto o dataset como a transcrição obtida pelo modelo ASR tenham a mesma formatação por isso ambos vão passar por um normalização simples de remoção de duplos espaços e conversão 
    de todas as palavras para lower case.
"""

dataset = []

for exemplo in json_file:
    """
        JOIN reconstrói a frase deixando a mesma apenas com espaços em branco normais. 
        SPLIT ajuda o JOIN, separando todas as palavras da frase.
        LOWER é auto explicativo.
    """
    dataset.append (" ".join(str(exemplo["trans"]).split()).lower())

display (dataset)

[{'audio_id': 1,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio1.wav',
  'trans': 'Boa tarde É aí da papelaria Sim O menina dá para me guardar dois bilhetes Dois bilhetes  Sim Que bilhetes diga-me Bilhetes lá do coiso que vai acontecer na Sexta Qual é o espetáculo diga-me É lá o que acontece lá no casino que o meu neto é que quer ir Mas eu não sei qual é Você sabe  É o da Sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam É assim mas você para comprar tem que vir cá pagá-los Sim está bem mas tem que mos guardar não é  Não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los Então e depois eu chego aí ',
  'duration': 42},
 {'audio_id': 2,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio2.wav',
  'trans': 'Boa tarde O m

['boa tarde é aí da papelaria sim o menina dá para me guardar dois bilhetes dois bilhetes sim que bilhetes diga-me bilhetes lá do coiso que vai acontecer na sexta qual é o espetáculo diga-me é lá o que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam é assim mas você para comprar tem que vir cá pagá-los sim está bem mas tem que mos guardar não é não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los então e depois eu chego aí',
 'boa tarde o menina até estou nervosa que vim de lá de baixo o menina eu já encontrei encontrou o quê encontrei a elisa a então eu vou passar aqui ás relações públicas e diz está bem só um momento relações públicas boa tarde olhe o menino eu já encontrei encontrou o quê a 

<h5> Model Loader </h5>

In [3]:
MODEL_PATH = r"C:\Users\Admin\Desktop\models\ASR Models\Whisper\WhisperLv3-PT-All 4Bit"

PROCESSOR = AutoProcessor.from_pretrained (MODEL_PATH)
MODEL = AutoModelForSpeechSeq2Seq.from_pretrained (MODEL_PATH, device_map = device, dtype = torch.float16)

W0826 10:13:42.419000 6052 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

<hr>

<h3> Avaliação Sistema Versão 1 </h3>

```mermaid 
flowchart LR
    ÁUDIO[Entrada do Áudio]
    ÁUDIO --> PROCESSOR[Tokenização do Áudio]
    PROCESSOR --> INFERÊNCIA[Encoding e Decoding\ndo Modelo ASR]
    INFERÊNCIA --> EVAL[Avaliação]


In [4]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt")

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

In [9]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[7.731836318969727, 9.599955558776855, 4.566570997238159, 7.220751523971558, 9.474120616912842, 7.8498406410217285, 4.175218343734741, 13.005277395248413, 4.583116769790649, 4.316499471664429, 5.332631587982178, 10.757428407669067, 6.732789754867554, 13.356438398361206]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora me porque é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêsim está bem mas tem que me os guardar não é',
 'e viu boa tardee ela acabou de la ela mora no trinta e doze seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trinta e dois seu pai em são paulo não mora no trint

{'WER_CALC': [0.47058823529411764, 1.2342342342342343, 0.5169491525423728, 0.5882352941176471, 0.32051282051282054, 0.2937062937062937, 0.7352941176470589, 0.4682926829268293, 0.4148936170212766, 0.5, 0.6375, 0.5675675675675675, 0.5748502994011976, 0.28110599078341014], 1: 0.47058823529411764, 2: 1.2342342342342343, 3: 0.5169491525423728, 4: 0.5882352941176471, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.7352941176470589, 8: 0.4682926829268293, 9: 0.4148936170212766, 10: 0.5, 11: 0.6375, 12: 0.5675675675675675, 13: 0.5748502994011976, 14: 0.28110599078341014}
{'CER_CALC': [0.42718446601941745, 0.785063752276867, 0.4731369150779896, 0.5366834170854271, 0.20874471086036672, 0.23427041499330656, 0.6646464646464646, 0.3577981651376147, 0.33410672853828305, 0.4050925925925926, 0.5487364620938628, 0.4518581081081081, 0.49238578680203043, 0.19518716577540107], 1: 0.42718446601941745, 2: 0.785063752276867, 3: 0.4731369150779896, 4: 0.5366834170854271, 5: 0.20874471086036672, 6: 0.23427

<p align = "center">
    <img src = "Screenshot 2026-08-26 102758.png">
</p>

<hr>

<h3> Avaliação Sistema Versão 2 </h3>

```mermaid 
flowchart LR
    ÁUDIO[Entrada do Áudio]
    --> PROCESSOR[Tokenização do Áudio]
    --> INFERÊNCIA[Encoding e Decoding do ASR\nBeam Search 5]
    --> EVAL[Avaliação]

In [10]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [11]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[12.77314567565918, 31.960999488830566, 5.9332520961761475, 9.679952383041382, 12.861374378204346, 10.437676429748535, 6.554807901382446, 12.560208559036255, 6.0690062046051025, 7.483024835586548, 14.382473707199097, 16.24435329437256, 12.20352840423584, 18.88250994682312]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá los sim está bem mas tem que mas guardar não é não tenho que vir cá ponho o o isso a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trin

{'WER_CALC': [0.33986928104575165, 1.9189189189189189, 0.5169491525423728, 0.5637254901960784, 0.32051282051282054, 0.2937062937062937, 0.4803921568627451, 0.45365853658536587, 0.4148936170212766, 0.2619047619047619, 0.1375, 0.44594594594594594, 0.4431137724550898, 0.271889400921659], 1: 0.33986928104575165, 2: 1.9189189189189189, 3: 0.5169491525423728, 4: 0.5637254901960784, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.4803921568627451, 8: 0.45365853658536587, 9: 0.4148936170212766, 10: 0.2619047619047619, 11: 0.1375, 12: 0.44594594594594594, 13: 0.4431137724550898, 14: 0.271889400921659}
{'CER_CALC': [0.23578363384188628, 1.4426229508196722, 0.4731369150779896, 0.5266331658291458, 0.20874471086036672, 0.23427041499330656, 0.43434343434343436, 0.36289500509684, 0.33410672853828305, 0.1712962962962963, 0.07821901323706378, 0.31334459459459457, 0.34390862944162437, 0.16934046345811052], 1: 0.23578363384188628, 2: 1.4426229508196722, 3: 0.4731369150779896, 4: 0.5266331658291458, 5

<p align = "center">
    <img src = "Screenshot 2026-08-26 103526.png">
</p>

<hr>

<h3> Avaliação Sistema Versão 3 </h3>

```mermaid 
flowchart LR
    ÁUDIO[Entrada do Áudio]
    --> PROCESSOR[Tokenização do Áudio]
    --> INFERÊNCIA[Encoding e Decoding do ASR\nBeam Search 10]
    --> EVAL[Avaliação]

In [12]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 10)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [13]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[14.716593265533447, 41.859153509140015, 6.982146263122559, 11.667519330978394, 16.755046844482422, 13.636707544326782, 9.04832148551941, 16.7338445186615, 7.932849884033203, 9.364268779754639, 18.481477975845337, 23.49244737625122, 17.110599756240845, 22.890239238739014]


['e foi boa tarde sim sim dois bilhotes sim bilhetes lá do do coiso que vai acontecer na sexta qual é o espetáculo me diz é lá o do que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu já liguei para aqui para dentro do sítio tenho que lá ir tenho que lá ir e agora disseram por que é que não liga lá para a papelaria que eles guardam oh é assim eu mas vocêcomprar tenho que vir cá e los sim está bem mas tem que mas guardar não é não tenho que vir cá e o o esse a tirar na máquina e sair da impressora ao papel para você me pagar na hora e não posso los então e depois eu cheguei a',
 'o menino eu estou nervosa que vim do lado de baixo o menino eu já encontrei encontrei a elisa o olhe o menino eu já encontrei a elisa que ela mora aqui em baixo vocês todas as noites perguntam uma vizinha minha já me tinha dito que ela mora ali em baixo temos que ligar para ela a avisare ela é que vou denunciá la ela mora no trinta

{'WER_CALC': [0.3464052287581699, 1.9189189189189189, 0.5169491525423728, 0.5637254901960784, 0.32051282051282054, 0.2937062937062937, 0.4803921568627451, 0.4585365853658537, 0.4148936170212766, 0.23809523809523808, 0.13125, 0.32882882882882886, 0.3712574850299401, 0.271889400921659], 1: 0.3464052287581699, 2: 1.9189189189189189, 3: 0.5169491525423728, 4: 0.5637254901960784, 5: 0.32051282051282054, 6: 0.2937062937062937, 7: 0.4803921568627451, 8: 0.4585365853658537, 9: 0.4148936170212766, 10: 0.23809523809523808, 11: 0.13125, 12: 0.32882882882882886, 13: 0.3712574850299401, 14: 0.271889400921659}
{'CER_CALC': [0.2302357836338419, 1.4426229508196722, 0.4731369150779896, 0.5266331658291458, 0.20874471086036672, 0.23427041499330656, 0.43434343434343436, 0.36289500509684, 0.33410672853828305, 0.16203703703703703, 0.06618531889290012, 0.19087837837837837, 0.2753807106598985, 0.16844919786096257], 1: 0.2302357836338419, 2: 1.4426229508196722, 3: 0.4731369150779896, 4: 0.5266331658291458, 5: 

<p align = "center">
    <img src = "Screenshot 2026-08-26 104205.png">
</p>

<hr>